# Fashion Assistant — Siamese Network Experiment

## What This Notebook Is For

This is the third experiment notebook in the Fashion Assistant project. The progression:

```
model_experiments.ipynb     →    fine_tuning.ipynb     →    siamese.ipynb
  Pre-trained baselines           Triplet fine-tuning        Siamese architecture
  (zero training)                 (adapt existing model)     (purpose-built for pairs)
```

The previous two notebooks used models that produce **independent embeddings** — each image is encoded separately, then similarity is computed via matrix multiplication. A Siamese Network is fundamentally different: it is designed from scratch to compare **pairs** of images.

---

## What is a Siamese Neural Network?

Named after Siamese twins — two identical networks joined together. Both share the exact same weights.

```
consumer photo ──→ [backbone] ──→ embedding_A ──┐
                   (shared                       ├──→ distance → loss
shop photo     ──→  weights)  ──→ embedding_B ──┘
```

**Why shared weights?** We want both branches to measure images in the same space. If the weights were different, "similar" would mean different things in each branch — comparisons would be meaningless.

**The training signal — Contrastive Loss:**

Instead of triplets (anchor, positive, negative), Siamese networks use **pairs** with a binary label:
- Label = 1 → same item (consumer + matching shop photo)
- Label = 0 → different items

```
Contrastive Loss:
  same pair   (label=1): loss = distance²                   ← pull closer
  diff pair   (label=0): loss = max(0, margin - distance)²  ← push apart
```

When a same-item pair is far apart → high loss → weights adjust to pull them closer.
When a different-item pair is already far enough apart (> margin) → loss = 0 → no update needed.

---

## Siamese vs Triplet Fine-Tuning — Key Differences

| | Triplet Fine-Tuning | Siamese Network |
|--|--|--|
| **Input per step** | 3 images (anchor, positive, negative) | 2 images (pair + label) |
| **Loss** | Triplet margin loss | Contrastive loss |
| **Architecture** | Single backbone, used 3x | Two branches, shared weights |
| **Inference** | Embed once → matmul search (fast) | Score per pair (slower at scale) |
| **Training data** | Triplets | Pairs with binary labels |
| **Best for** | Large-scale retrieval | High-precision pair verification |

**In production:** Siamese is often used as a **re-ranker** — first use fast embedding search to get top-20 candidates, then use Siamese to precisely re-score those 20 pairs. We will test this pattern here.

---

## Backbone Choice

We use **EfficientNet-B4** as the shared backbone, loaded from the fine-tuned weights saved in `fine_tuning.ipynb`. This choice was made after evaluating all four models:

| Model | Recall@1 (baseline) | Siamese compatibility |
|---|---|---|
| ResNet-50 | 3.08% | ✅ CNN, euclidean-friendly |
| **EfficientNet-B4** | **3.89%** | **✅ Best baseline + CNN** |
| DINOv2 ViT-B/14 | higher | ⚠️ Large, risk of forgetting with small data |
| CLIP ViT-B/32 | highest | ❌ Trained with cosine similarity — conflicts with contrastive loss |

**Why EfficientNet-B4 specifically:**

1. **Best baseline performance** — Recall@1 of 3.89% vs ResNet-50's 3.08% in `model_experiments.ipynb`. Starting from a stronger feature extractor gives the Siamese network a better foundation.

2. **CNN features are euclidean-friendly** — Contrastive loss minimizes euclidean distance between same-item pairs. CNN features have no pre-imposed similarity metric, so the loss can freely shape the embedding space. CLIP, by contrast, was pre-trained with cosine similarity — contrastive loss conflicts with this and degrades retrieval quality (observed: Recall@1 dropped to 0.89% when CLIP was used as Siamese backbone).

3. **High-resolution inputs (380×380)** — EfficientNet-B4's larger input captures fine-grained fashion details: fabric texture, stitching, print patterns — critical for distinguishing similar clothing items.

4. **Already fine-tuned on fashion triplets** — The `efficientnet_b4_best.pth` weights were trained on DeepFashion consumer-to-shop pairs, so the features are already adapted to the fashion retrieval domain before Siamese training begins.

5. **Appropriate size** — Larger than ResNet-50 (17.5M vs 25.6M trainable params) but smaller than DINOv2 or full CLIP, making Siamese fine-tuning feasible with a 5% data subset without catastrophic forgetting.

---

## Evaluation Plan

Two modes:
1. **Standalone retrieval** — use Siamese embeddings directly for gallery search (compare to triplet fine-tuned ResNet-50)
2. **Re-ranker** — use fast matmul to get top-50 candidates, then Siamese to re-score and re-rank (tests the production pattern)

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from datasets import FashionDataset, PairFashionDataset

DATASET_PATH   = "/Users/anushrin/Documents/Cowork Playground/FashionRecommendation/workspace/Capstone Fashion Assistant/fashion_assistant/data/DeepFashion Database/Data/Consumer-to-shop Clothes Retrieval Benchmark"
EMBEDDINGS_DIR = "/Users/anushrin/Documents/Cowork Playground/FashionRecommendation/workspace/Capstone Fashion Assistant/fashion_assistant/embeddings"
MODELS_DIR     = "/Users/anushrin/Documents/Cowork Playground/FashionRecommendation/workspace/Capstone Fashion Assistant/fashion_assistant/models"
os.makedirs(MODELS_DIR, exist_ok=True)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: mps


---
## Section 1 — Load Data

In [2]:
eval_path = os.path.join(DATASET_PATH, "Eval", "list_eval_partition.txt")
df_eval   = pd.read_csv(eval_path, sep=r'\s+', skiprows=1)
df_eval.columns = ['consumer_image', 'shop_image', 'item_id', 'split']

print("Available splits:", df_eval['split'].unique())

df_train = df_eval[df_eval['split'] == 'train'].reset_index(drop=True)
df_test  = df_eval[df_eval['split'] == 'test'].reset_index(drop=True)
df_val   = df_eval[df_eval['split'] == 'val'].reset_index(drop=True)

if len(df_val) == 0:
    val_size = int(0.1 * len(df_train))
    df_val   = df_train.iloc[:val_size].reset_index(drop=True)
    df_train = df_train.iloc[val_size:].reset_index(drop=True)
    print("No 'val' split found — carved 10% from train.")

gallery_paths   = df_test['shop_image'].unique().tolist()
gallery_index   = {path: i for i, path in enumerate(gallery_paths)}
query_paths     = df_test['consumer_image'].tolist()
query_index_map = {path: i for i, path in enumerate(query_paths)}

print(f"Train pairs: {len(df_train)}")
print(f"Val pairs:   {len(df_val)}")
print(f"Test pairs:  {len(df_test)}")
print(f"Gallery:     {len(gallery_paths)} unique shop images")

Available splits: <StringArray>
['test', 'train', 'val']
Length: 3, dtype: str
Train pairs: 98832
Val pairs:   48935
Test pairs:  47773
Gallery:     11312 unique shop images


---
## Section 2 — Transforms and Shared Utilities

In [3]:
from torchvision.models import efficientnet_b4

# EfficientNet-B4 transform (380×380, same as fine_tuning.ipynb)
efficientnet_transform = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Gallery / query index maps for Recall@K evaluation
gallery_paths   = df_test['shop_image'].unique().tolist()
gallery_index   = {path: i for i, path in enumerate(gallery_paths)}
query_paths     = df_test['consumer_image'].tolist()
query_index_map = {path: i for i, path in enumerate(query_paths)}

print(f"Gallery: {len(gallery_paths)} shop images")
print(f"Queries: {len(query_paths)} consumer images")

def evaluate_recall_at_k(query_embeddings, gallery_embeddings, k_values=[1, 5, 10, 20]):
    mask            = df_test['consumer_image'].isin(query_index_map) & df_test['shop_image'].isin(gallery_index)
    df_valid        = df_test[mask]
    q_indices       = df_valid['consumer_image'].map(query_index_map).values
    correct_indices = df_valid['shop_image'].map(gallery_index).values
    gal  = gallery_embeddings.astype(np.float32)
    hits = {k: 0 for k in k_values}
    for start in tqdm(range(0, len(q_indices), 1000), desc="Evaluating"):
        end           = min(start + 1000, len(q_indices))
        scores        = query_embeddings[q_indices[start:end]].astype(np.float32) @ gal.T
        top_k_results = np.argsort(-scores, axis=1)[:, :max(k_values)]
        for i, correct_idx in enumerate(correct_indices[start:end]):
            for k in k_values:
                if correct_idx in top_k_results[i][:k]:
                    hits[k] += 1
    total  = len(q_indices)
    recall = {f"Recall@{k}": round(hits[k] / total * 100, 2) for k in k_values}
    recall['total_queries'] = total
    return recall

---
## Section 3 — Siamese Network Architecture

### How the Architecture Works

```
SiameseNetwork
├── backbone (ResNet-50, classification head removed)
│     shared between both branches — same weights, same forward pass
└── forward(img1, img2)
      emb1 = backbone(img1)  → L2 normalize
      emb2 = backbone(img2)  → L2 normalize
      distance = euclidean(emb1, emb2)
      return emb1, emb2, distance
```

**Why return both embeddings AND distance?**
- During training: we use `distance` to compute contrastive loss
- During evaluation (standalone retrieval): we use `emb1`/`emb2` for gallery search
- During evaluation (re-ranker): we use `distance` to re-score candidates

**Why L2 normalize inside the network?**
Normalization ensures the distance is purely about direction (cosine), not magnitude. Without it, the model could cheat by shrinking all embeddings toward zero (low distance = low loss, but meaningless).

In [4]:
class SiameseNetwork(nn.Module):
    """Siamese network using fine-tuned EfficientNet-B4 as shared backbone (1792-dim)."""
    def __init__(self):
        super().__init__()
        base = efficientnet_b4(weights=None)
        base.classifier = nn.Identity()   # output: (batch, 1792)
        self.backbone = base

    def encode(self, x):
        emb = self.backbone(x)
        return nn.functional.normalize(emb, p=2, dim=1)  # L2 normalize

    def forward(self, img1, img2):
        emb1 = self.encode(img1)
        emb2 = self.encode(img2)
        distance = torch.nn.functional.pairwise_distance(emb1, emb2)
        return emb1, emb2, distance


siamese = SiameseNetwork().to(DEVICE)

# Load fine-tuned EfficientNet-B4 weights
ckpt_path = os.path.join(MODELS_DIR, "efficientnet_b4_best.pth")
state = torch.load(ckpt_path, map_location=DEVICE)
# Fine-tuning saved with classifier=Identity, so keys match directly
siamese.backbone.load_state_dict(state, strict=False)
print(f"Loaded fine-tuned EfficientNet-B4 weights from {ckpt_path}")

total = sum(p.numel() for p in siamese.parameters())
print(f"Total parameters: {total:,}")

Total parameters: 23,508,032


---
## Section 4 — Contrastive Loss

### Why a Custom Loss?

PyTorch doesn't have a built-in `ContrastiveLoss`, so we write it ourselves. The formula:

```
For a same-item pair   (label = 1):  loss = distance²
For a diff-item pair   (label = 0):  loss = max(0, margin - distance)²

Combined:
loss = label × d² + (1 - label) × max(0, margin - d)²
```

**`max(0, margin - distance)`** — if the negative pair is already separated by more than `margin`, the loss is zero. We only penalize if wrong pairs are too close together.

**`margin`** — typically 1.0 for euclidean distance on normalized vectors. Determines how far apart negative pairs must be pushed.

In [5]:
class ContrastiveLoss(nn.Module):
    """
    Computes contrastive loss for Siamese networks based on pairwise distances.
    
    For positive pairs (same item), minimizes squared distance.
    For negative pairs (different items), penalizes distances less than the margin.
    """
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, distance, label):
        # label=1 → same item, label=0 → different item
        same_loss = label       * distance.pow(2)
        diff_loss = (1 - label) * torch.clamp(self.margin - distance, min=0).pow(2)
        return (same_loss + diff_loss).mean()

---
## Section 5 — Pair Dataset

Unlike triplets (3 images), contrastive learning uses **pairs** with a binary label.

**Positive pair** (label=1): consumer photo + its matching shop photo — directly from `df_train`.

**Negative pair** (label=0): consumer photo + a shop photo from a *different* item — randomly sampled.

We generate one negative for every positive to keep the dataset balanced (50% same, 50% different).

In [6]:
# PairFashionDataset is imported from datasets.py
# See datasets.py for implementation details:
#   first half of indices  → positive pairs (label=1)
#   second half            → negative pairs (label=0)
#   total length = 2 × len(df)
print("PairFashionDataset ready:", PairFashionDataset)

PairFashionDataset ready: <class 'datasets.PairFashionDataset'>


---
## Section 6 — Training Loop

### What Happens Each Epoch

```
For each batch of (img1, img2, label):
  1. Forward pass through both branches → emb1, emb2, distance
  2. Compute contrastive loss using distance + label
  3. Backpropagate — gradients flow through both branches simultaneously
  4. Update shared weights once — both branches update together
```

**Why do both branches update from one backward pass?**
Because they share weights — there's only one set of parameters. The gradient from both `emb1` and `emb2` paths accumulates into the same weight tensors.

In [ ]:
EPOCHS     = 5
BATCH_SIZE = 16     # EfficientNet-B4 at 380×380 is memory-heavy
LR         = 1e-4
MARGIN     = 1.0
PATIENCE   = 2      # early stopping patience

# 25% subset — same strategy as fine_tuning.ipynb for faster training
df_train_sub = df_train.sample(frac=0.05, random_state=42).reset_index(drop=True)
print(f"Using {len(df_train_sub)} / {len(df_train)} training pairs (5%)")

train_loader = DataLoader(
    PairFashionDataset(df_train_sub, DATASET_PATH, efficientnet_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    PairFashionDataset(df_val, DATASET_PATH, efficientnet_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

optimizer = optim.Adam(siamese.parameters(), lr=LR)
loss_fn   = ContrastiveLoss(margin=MARGIN)

train_losses          = []
val_losses            = []
best_val_loss         = float('inf')
epochs_no_improvement = 0

for epoch in range(EPOCHS):
    # --- Training ---
    siamese.train()
    total_train_loss = 0.0
    for img1, img2, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]"):
        img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE)
        _, _, distance = siamese(img1, img2)
        loss = loss_fn(distance, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    # --- Validation ---
    siamese.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for img1, img2, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  "):
            img1, img2, labels = img1.to(DEVICE), img2.to(DEVICE), labels.to(DEVICE)
            _, _, distance = siamese(img1, img2)
            total_val_loss += loss_fn(distance, labels).item()

    avg_train = total_train_loss / len(train_loader)
    avg_val   = total_val_loss   / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    # Save every epoch checkpoint
    epoch_path = os.path.join(MODELS_DIR, f"siamese_epoch{epoch+1}.pth")
    torch.save(siamese.state_dict(), epoch_path)
    print(f"Epoch {epoch+1} — train: {avg_train:.4f} | val: {avg_val:.4f} | saved → {os.path.basename(epoch_path)}", end="")

    if avg_val < best_val_loss:
        best_val_loss         = avg_val
        epochs_no_improvement = 0
        torch.save(siamese.state_dict(), os.path.join(MODELS_DIR, "siamese_best.pth"))
        print("  ← best", end="")
    else:
        epochs_no_improvement += 1
        print(f"  (no improvement {epochs_no_improvement}/{PATIENCE})", end="")
    print()

    if epochs_no_improvement >= PATIENCE:
        print(f"Early stopping — val loss did not improve for {PATIENCE} epochs.")
        break

print(f"\nBest val loss: {best_val_loss:.4f} — siamese_best.pth")

# --- Loss curve ---
actual_epochs = len(train_losses)
plt.figure(figsize=(8, 4))
plt.plot(range(1, actual_epochs+1), train_losses, marker='o', label='Train Loss', color='steelblue')
plt.plot(range(1, actual_epochs+1), val_losses,   marker='o', label='Val Loss',   color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('Contrastive Loss')
plt.title('Siamese Network — Train vs Val Loss per Epoch')
plt.xticks(range(1, actual_epochs+1))
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "siamese_loss_curve.png"))
plt.show()

Using 24708 / 98832 training pairs (25%)


Epoch 1/5 [train]:  19%|█▊        | 72/387 [2:42:29<11:14:50, 128.54s/it]

---
## Section 7 — Evaluation

### Mode 1: Standalone Retrieval

Use the Siamese backbone to extract embeddings for all gallery and query images, then run the same Recall@K evaluation as all other notebooks. This gives a direct, fair comparison.

In [ ]:
# Siamese (EfficientNet-B4 backbone) — extract embeddings or load from disk
siamese.eval()

def extract_siamese_embeddings(image_paths, batch_size=32, desc="Siamese-EffNet"):
    dataset = FashionDataset(image_paths, DATASET_PATH, efficientnet_transform)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    embeddings = np.zeros((len(image_paths), 1792), dtype=np.float32)  # EfficientNet-B4 = 1792-dim
    with torch.no_grad():
        for imgs, idxs in tqdm(loader, desc=desc):
            emb = siamese.encode(imgs.to(DEVICE)).cpu().numpy()
            embeddings[idxs] = emb
    return embeddings

gallery_path_npy = os.path.join(EMBEDDINGS_DIR, "trained_siamese_effnet_gallery.npy")
query_path_npy   = os.path.join(EMBEDDINGS_DIR, "trained_siamese_effnet_queries.npy")

if os.path.exists(gallery_path_npy) and os.path.exists(query_path_npy):
    print("Found saved embeddings — loading from disk.")
    gallery_emb_siam = np.load(gallery_path_npy)
    query_emb_siam   = np.load(query_path_npy)
else:
    print("No saved embeddings — extracting now.")
    gallery_emb_siam = extract_siamese_embeddings(gallery_paths, desc="Siamese-EffNet Gallery")
    query_emb_siam   = extract_siamese_embeddings(query_paths,   desc="Siamese-EffNet Queries")
    np.save(gallery_path_npy, gallery_emb_siam)
    np.save(query_path_npy,   query_emb_siam)
    print("Embeddings saved.")

results_siamese = evaluate_recall_at_k(query_emb_siam, gallery_emb_siam)
print("Siamese Network (EfficientNet-B4 backbone) Results:")
for k, v in results_siamese.items():
    print(f"  {k}: {v}")

### Mode 2: Siamese as Re-Ranker

This is the **production pattern** for high-accuracy retrieval systems:

```
Step 1 — Fast candidate retrieval:
  query embedding @ gallery.T → top-50 candidates (numpy matmul, milliseconds)

Step 2 — Precise re-scoring:
  For each of the 50 candidates:
    siamese.forward(query_img, candidate_img) → distance
  Re-rank by siamese distance

Step 3 — Final Recall@K on re-ranked results
```

**Why this works better than standalone:** The fast first step filters 11,312 gallery images to 50 plausible candidates. The Siamese then applies precise per-pair scoring only on those 50 — getting fine-grained comparison without the cost of scoring all 11,312 pairs.

In [ ]:
def evaluate_with_reranker(query_embeddings, gallery_embeddings,
                            candidate_k=50, k_values=[1, 5, 10, 20]):
    mask            = df_test['consumer_image'].isin(query_index_map) & df_test['shop_image'].isin(gallery_index)
    df_valid        = df_test[mask]
    q_indices       = df_valid['consumer_image'].map(query_index_map).values[:500]  # limit for speed
    q_paths         = df_valid['consumer_image'].values[:500]
    correct_indices = df_valid['shop_image'].map(gallery_index).values[:500]

    gal  = gallery_embeddings.astype(np.float32)
    hits = {k: 0 for k in k_values}

    siamese.eval()
    for start in tqdm(range(0, len(q_indices), 100), desc="Re-ranking"):
        end = min(start + 100, len(q_indices))

        # Step 1: fast candidate retrieval
        q_batch  = query_embeddings[q_indices[start:end]].astype(np.float32)
        scores   = q_batch @ gal.T
        top_candidates = np.argsort(-scores, axis=1)[:, :candidate_k]

        # Step 2: Siamese re-scoring
        for i in range(end - start):
            q_path   = q_paths[start + i]
            try:
                q_img = imagenet_transform(
                    Image.open(os.path.join(DATASET_PATH, q_path)).convert("RGB")
                ).unsqueeze(0).to(DEVICE)
            except Exception:
                q_img = torch.zeros(1, 3, 224, 224).to(DEVICE)

            candidate_idxs = top_candidates[i]
            distances = []
            with torch.no_grad():
                for cand_idx in candidate_idxs:
                    try:
                        c_img = imagenet_transform(
                            Image.open(os.path.join(DATASET_PATH, gallery_paths[cand_idx])).convert("RGB")
                        ).unsqueeze(0).to(DEVICE)
                    except Exception:
                        c_img = torch.zeros(1, 3, 224, 224).to(DEVICE)
                    _, _, dist = siamese(q_img, c_img)
                    distances.append(dist.item())

            # Re-rank by siamese distance (smaller = more similar)
            reranked = candidate_idxs[np.argsort(distances)]

            correct_idx = correct_indices[start + i]
            for k in k_values:
                if correct_idx in reranked[:k]:
                    hits[k] += 1

    total  = len(q_indices)
    recall = {f"Recall@{k}": round(hits[k] / total * 100, 2) for k in k_values}
    recall['total_queries'] = total
    return recall


# Load ResNet-50 baseline embeddings for the fast first-pass retrieval
gallery_emb_base = np.load(os.path.join(EMBEDDINGS_DIR, "resnet50_gallery.npy"))
query_emb_base   = np.load(os.path.join(EMBEDDINGS_DIR, "resnet50_queries.npy"))

results_reranked = evaluate_with_reranker(query_emb_base, gallery_emb_base)
print("Siamese Re-Ranker Results:")
for k, v in results_reranked.items():
    print(f"  {k}: {v}")

---
## Section 8 — Full Comparison

All approaches on the same test set.

In [ ]:
# Load results from saved embeddings for final comparison
gallery_emb_resnet = np.load(os.path.join(EMBEDDINGS_DIR, "resnet50_gallery.npy"))
query_emb_resnet   = np.load(os.path.join(EMBEDDINGS_DIR, "resnet50_queries.npy"))
results_resnet_base = evaluate_recall_at_k(query_emb_resnet, gallery_emb_resnet)

# CLIP fine-tuned — best model from fine_tuning.ipynb
gallery_emb_clip_ft = np.load(os.path.join(EMBEDDINGS_DIR, "trained_clip_gallery.npy"))
query_emb_clip_ft   = np.load(os.path.join(EMBEDDINGS_DIR, "trained_clip_queries.npy"))
results_clip_ft = evaluate_recall_at_k(query_emb_clip_ft, gallery_emb_clip_ft)

comparison = pd.DataFrame([
    {'Model': 'ResNet-50 baseline',   **{k: v for k, v in results_resnet_base.items() if k != 'total_queries'}},
    {'Model': 'CLIP fine-tuned',       **{k: v for k, v in results_clip_ft.items()     if k != 'total_queries'}},
    {'Model': 'Siamese standalone',    **{k: v for k, v in results_siamese.items()     if k != 'total_queries'}},
    {'Model': 'Siamese re-ranker',     **{k: v for k, v in results_reranked.items()    if k != 'total_queries'}},
])

print(comparison.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
k_labels = ['Recall@1', 'Recall@5', 'Recall@10', 'Recall@20']
x        = np.arange(len(k_labels))
width    = 0.2
colors   = ['steelblue', 'seagreen', 'darkorange', 'crimson']

for i, (_, row) in enumerate(comparison.iterrows()):
    ax.bar(x + i * width, [row[k] for k in k_labels], width,
           label=row['Model'], color=colors[i], alpha=0.85)

ax.set_xlabel('Metric')
ax.set_ylabel('Recall (%)')
ax.set_title('Siamese vs CLIP Fine-Tuned vs Baseline')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(k_labels)
ax.legend()
plt.tight_layout()
plt.show()

### My Analysis

*(Fill in after running — answer in first person)*

- Did the Siamese network outperform the triplet fine-tuned ResNet-50?
- Did the re-ranker improve over standalone Siamese? By how much?
- What is the speed trade-off of the re-ranker vs standalone retrieval?
- Which approach would you recommend for production and why?
- What would you try next to improve Siamese performance further?